
# DS2002 — SQL + Python → Pandas DataFrames
## Studio — Working with DataFrames

This studio is designed to be completed **independently**.

You will:
- pull structured data from SQL into a DataFrame
- answer questions using Pandas (not SQL)
- practice filtering, sorting, grouping, and feature creation

The goal is not to memorize syntax.
The goal is to become comfortable *thinking with DataFrames*.



### Submission (Canvas)

1. Complete all TODOs below.
2. Run every cell so outputs are visible.
3. Save your Kaggle notebook.
4. Submit the **Kaggle Notebook GitHub URL** in Canvas.



## Part 0 — Setup (Run This Cell)

This cell prepares SQLite, Pandas, and helper functions.


In [6]:

import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

def exec_sql(sql):
    conn.executescript(sql)
    conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)

print("Environment ready.")


Environment ready.



## Part 1 — Create and Load the Store Database (Provided)

Run the next cell to create tables and insert data.


In [7]:

exec_sql('''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    region TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    price REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT
);

CREATE TABLE order_items (
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER
);

INSERT INTO customers VALUES
(1,'Alice','East'),
(2,'Bob','West'),
(3,'Carol','East'),
(4,'David','South');

INSERT INTO products VALUES
(101,'Keyboard','Electronics',50),
(102,'Mouse','Electronics',20),
(103,'Monitor','Electronics',200),
(104,'Desk','Furniture',300),
(105,'Chair','Furniture',150);

INSERT INTO orders VALUES
(1,1,'2026-02-01'),
(2,2,'2026-02-02'),
(3,1,'2026-02-03'),
(4,3,'2026-02-03'),
(5,4,'2026-02-04');

INSERT INTO order_items VALUES
(1,101,1),(1,102,2),
(2,103,1),
(3,104,1),(3,105,1),
(4,101,2),(4,103,1),
(5,105,2);
''')

print("Store data loaded.")


Store data loaded.



## Part 2 — Pull Data into a DataFrame

You will use **one SQL query** to create a DataFrame.
After this point, **do not use SQL** to answer questions.


In [8]:

df = q('''
SELECT
    c.name AS customer,
    c.region,
    o.order_id,
    o.order_date,
    p.name AS product,
    p.category,
    oi.quantity,
    p.price,
    (oi.quantity * p.price) AS line_total
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
ORDER BY o.order_date;
''')

df


,customer,region,order_id,order_date,product,category,quantity,price,line_total
0,Alice,East,1,2026-02-01,Keyboard,Electronics,1,50.0,50.0
1,Alice,East,1,2026-02-01,Mouse,Electronics,2,20.0,40.0
2,Bob,West,2,2026-02-02,Monitor,Electronics,1,200.0,200.0
3,Alice,East,3,2026-02-03,Desk,Furniture,1,300.0,300.0
4,Alice,East,3,2026-02-03,Chair,Furniture,1,150.0,150.0
5,Carol,East,4,2026-02-03,Keyboard,Electronics,2,50.0,100.0
6,Carol,East,4,2026-02-03,Monitor,Electronics,1,200.0,200.0
7,David,South,5,2026-02-04,Chair,Furniture,2,150.0,300.0



## Part 3 — DataFrame Questions

Answer each question using **Pandas operations only**.


### Q1. How many rows and columns are in the DataFrame?

In [9]:

# TODO
# print(df.shape)

print(df.shape[0])## num of rows 
print(df.shape[1])##num of columns


8
9


### Q2. List the column names and their data types.

In [10]:

# TODO

pd.DataFrame({
    "Names": df.columns,
    "DataType": df.dtypes.values
})


,Names,DataType
0,customer,object
1,region,object
2,order_id,int64
3,order_date,object
4,product,object
5,category,object
6,quantity,int64
7,price,float64
8,line_total,float64


Q3. Show only the rows where the customer is Alice.

In [11]:

# TODO
df_customers = q("SELECT * FROM customers")
# Check the column names

alice = df_customers[df_customers['name'] == 'Alice']
print(alice)


   customer_id   name region
0            1  Alice   East


### Q4. Show all line items with a line total greater than $100.

In [12]:
# TODO
df_order_items = q("SELECT * FROM order_items")
df_products = q("SELECT * FROM products")

df_line_items = df_order_items.merge(df_products, on="product_id")

df_line_items['line_total'] = df_line_items['quantity'] * df_line_items['price']


df_expensive = df_line_items[df_line_items['line_total'] > 100]
print(df_expensive)


   order_id  product_id  quantity     name     category  price  line_total
2         2         103         1  Monitor  Electronics  200.0       200.0
3         3         104         1     Desk    Furniture  300.0       300.0
4         3         105         1    Chair    Furniture  150.0       150.0
6         4         103         1  Monitor  Electronics  200.0       200.0
7         5         105         2    Chair    Furniture  150.0       300.0


###Q5. Sort the DataFrame by line_total from highest to lowest.

In [13]:
# TODO

df_sorted = df_line_items[df_line_items['line_total'] > 100].sort_values( by='line_total', ascending = False)
print(df_sorted)


   order_id  product_id  quantity     name     category  price  line_total
3         3         104         1     Desk    Furniture  300.0       300.0
7         5         105         2    Chair    Furniture  150.0       300.0
2         2         103         1  Monitor  Electronics  200.0       200.0
6         4         103         1  Monitor  Electronics  200.0       200.0
4         3         105         1    Chair    Furniture  150.0       150.0


### Q6. Create a new column called tax (7% of line_total).

In [14]:
# TODO
df_line_items['tax'] = df_line_items['line_total'] * 0.07
print(df_line_items)


   order_id  product_id  quantity      name     category  price  line_total  \
0         1         101         1  Keyboard  Electronics   50.0        50.0   
1         1         102         2     Mouse  Electronics   20.0        40.0   
2         2         103         1   Monitor  Electronics  200.0       200.0   
3         3         104         1      Desk    Furniture  300.0       300.0   
4         3         105         1     Chair    Furniture  150.0       150.0   
5         4         101         2  Keyboard  Electronics   50.0       100.0   
6         4         103         1   Monitor  Electronics  200.0       200.0   
7         5         105         2     Chair    Furniture  150.0       300.0   

    tax  
0   3.5  
1   2.8  
2  14.0  
3  21.0  
4  10.5  
5   7.0  
6  14.0  
7  21.0  


### Q7. Create a new column called total_with_tax.

In [15]:
# TODO
df_line_items['total_with_tax'] = df_line_items['line_total'] + df_line_items['tax']
print(df_line_items)


   order_id  product_id  quantity      name     category  price  line_total  \
0         1         101         1  Keyboard  Electronics   50.0        50.0   
1         1         102         2     Mouse  Electronics   20.0        40.0   
2         2         103         1   Monitor  Electronics  200.0       200.0   
3         3         104         1      Desk    Furniture  300.0       300.0   
4         3         105         1     Chair    Furniture  150.0       150.0   
5         4         101         2  Keyboard  Electronics   50.0       100.0   
6         4         103         1   Monitor  Electronics  200.0       200.0   
7         5         105         2     Chair    Furniture  150.0       300.0   

    tax  total_with_tax  
0   3.5            53.5  
1   2.8            42.8  
2  14.0           214.0  
3  21.0           321.0  
4  10.5           160.5  
5   7.0           107.0  
6  14.0           214.0  
7  21.0           321.0  


### Q8. What is the total revenue per customer?

In [16]:
# TODO
# Merge order_items with products
df_line_items = df_order_items.merge(q("SELECT * FROM products"), on="product_id")

# Compute line_total, tax, and total_with_tax
df_line_items['line_total'] = df_line_items['price'] * df_line_items['quantity']
df_line_items['tax'] = df_line_items['line_total'] * 0.07
df_line_items['total_with_tax'] = df_line_items['line_total'] + df_line_items['tax']

# Merge with orders to get customer_id
df_orders_items = df_line_items.merge(q("SELECT * FROM orders"), on="order_id")

# Merge with customers to get customer name
df_orders_items = df_orders_items.merge(q("SELECT customer_id, name as customer_name FROM customers"), on="customer_id")

# Group by the customer_name column
revenue_per_customer = df_orders_items.groupby('customer_name')['total_with_tax'].sum().reset_index()
revenue_per_customer.columns = ['Customer', 'TotalRevenue']

print(revenue_per_customer)



  Customer  TotalRevenue
0    Alice         577.8
1      Bob         214.0
2    Carol         321.0
3    David         321.0


### Q9. What is the average line_total per product category?

In [18]:
 #TODO

avg_line_total_per_category = df_line_items.groupby('category')['line_total'].mean().reset_index()
avg_line_total_per_category.columns = ['Category', 'AverageLineTotal']

print(avg_line_total_per_category)

      Category  AverageLineTotal
0  Electronics             118.0
1    Furniture             250.0


### Q10. Which customer spent the most overall?

In [19]:

# TODO
top_customer = revenue_per_customer.sort_values(by='TotalRevenue', ascending = False).head(1)

print(top_customer)


  Customer  TotalRevenue
0    Alice         577.8


### Q11. Which region generated the most revenue?

In [20]:

# TODO
# Merge line items with orders and customers to get region info
df_orders_items = df_line_items.merge(q("SELECT * FROM orders"), on="order_id")
df_orders_items = df_orders_items.merge(q("SELECT * FROM customers"), on="customer_id")

# Group by region and calculate total revenue
revenue_per_region = df_orders_items.groupby('region')['total_with_tax'].sum().reset_index()
revenue_per_region.columns = ['Region', 'TotalRevenue']

# Find the region with the highest revenue
top_region = revenue_per_region.sort_values(by='TotalRevenue', ascending=False).head(1)

print(top_region)



  Region  TotalRevenue
0   East         898.8


### Q12. Create a summary DataFrame with customer and total_spent.

In [21]:

# TODO
summary_df = df_orders_items.groupby('name_y')['total_with_tax'].sum().reset_index()
summary_df.columns = ['Customer', 'TotalSpent']

print(summary_df)


  Customer  TotalSpent
0    Alice       577.8
1      Bob       214.0
2    Carol       321.0
3    David       321.0


### Q13. Write that summary DataFrame back to the database as customer_summary.

In [23]:

# TODO
summary_df.to_sql('customer_summary', conn, if_exists='replace', index=False)


q("SELECT * FROM customer_summary")


,Customer,TotalSpent
0,Alice,577.8
1,Bob,214.0
2,Carol,321.0
3,David,321.0



## Part 4 — Reflection

In 2–3 sentences:
- What DataFrame operation felt most intuitive?
- What still feels confusing?


In [25]:

reflection = "Filtering rows and viewing columns felt most alot because it is similar to organizing data in a table. Joins and grouping still feel confusing because I have to think carefully about how rows match together and how the final result is calculated."

print(reflection)


Filtering rows and viewing columns felt most alot because it is similar to organizing data in a table. Joins and grouping still feel confusing because I have to think carefully about how rows match together and how the final result is calculated.



## You Are Done

Make sure:
- all TODOs are completed
- all cells run without error
- outputs are visible

Then submit your Kaggle Notebook Github URL in Canvas to me and Daniel
